# How much is a grid place worth, and where is a grid penalty cheapest?
Formula 1 race results 2014 to 2025 from F1DB (https://github.com/f1db/f1db, CC BY 4.0).

Run from the project folder:  `python src/get_data.py`  then  `python src/analysis.py`

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
DATA, FIG, RES = ROOT / "data" / "f1db", ROOT / "figures", ROOT / "results"
FIG.mkdir(exist_ok=True), RES.mkdir(exist_ok=True)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8982"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e4dd", "axes.axisbelow": True,
                     "font.size": 10.5, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})
FIRST, LAST, TEST_FROM = 2014, 2025, 2023       # hybrid-engine era; 2026 is excluded (season in progress, new rules)
funnel = []


def log(label, df, note):
    funnel.append({"label": label, "rows": int(len(df)), "note": note})
    print(f"{label:<45s} {len(df):>7,d} rows   {note}")

## 1. Load

In [2]:
res = pd.read_csv(DATA / "f1db-races-race-results.csv", low_memory=False)
races = pd.read_csv(DATA / "f1db-races.csv", low_memory=False)[["id", "date", "grandPrixId", "circuitId", "circuitType"]].rename(columns={"id": "raceId"})
log("Raw race results, 1950 onwards", res, "one row = one car entered in one race")
df = res[res["year"].between(FIRST, LAST)].merge(races, on="raceId", how="left", validate="many_to_one")
log(f"Seasons {FIRST}-{LAST}", df, "one rule set for points (top 10) and one engine formula")

Raw race results, 1950 onwards                 27,731 rows   one row = one car entered in one race
Seasons 2014-2025                               5,108 rows   one rule set for points (top 10) and one engine formula


## 2. Clean
### 2.1 The finishing position column is text, and the text is the story
`positionText` holds `1`…`22` for classified finishers and codes for everyone else: `DNF` (retired),
`DNS` (did not start), `DSQ` (disqualified), `NC` (not classified). `pd.to_numeric(errors="coerce")`
turns all of those into NaN. That is correct here, but only because each code is handled on purpose below.

In [3]:
print(df["positionText"].value_counts().loc[lambda s: ~s.index.str.isdigit()])
pos = pd.to_numeric(df["positionText"], errors="coerce")
print("Cells that coerce turned into NaN:", int(pos.isna().sum()), "of", len(df), f"({pos.isna().mean():.1%})")
df["finish"] = pos
df["status"] = np.where(pos.notna(), "classified", df["positionText"])

# Did-not-start is not a race outcome: the car never took the grid. Remove, and say so.
dns = df["status"].isin(["DNS", "DNP"])
df = df[~dns].copy()
log("Cars that started", df, f"{int(dns.sum())} did-not-start entries removed (no race to predict)")

# A retirement is NOT a missing value. It is a result: zero points. Dropping DNFs would delete the
# 14% of starts that went wrong and make every grid slot look safer than it is.
df["points_finish"] = (df["finish"] <= 10).astype(int)          # NaN <= 10 is False, which is exactly right
# `points` is empty for cars that scored nothing. Here an empty cell means 0, by design, so fillna(0) is right.
# (Without this line every average below would silently be "average among scorers only".)
print("Empty points cells set to 0:", int(df["points"].isna().sum()))
df["points"] = df["points"].fillna(0)
df["dnf"] = (df["status"] != "classified").astype(int)
print(f"Retired / disqualified / unclassified: {df['dnf'].mean():.1%} of starts;  points finishes: {df['points_finish'].mean():.1%}")

positionText
DNF    710
DNS     37
DSQ     18
DNP      2
NC       1
Name: count, dtype: int64
Cells that coerce turned into NaN: 768 of 5108 (15.0%)
Cars that started                               5,069 rows   39 did-not-start entries removed (no race to predict)
Empty points cells set to 0: 2549
Retired / disqualified / unclassified: 14.4% of starts;  points finishes: 49.7%


### 2.2 Grid position: pit-lane starts have no number
`gridPositionText == "PL"` means the car started from the pit lane, behind the whole field.
Leaving it as NaN would let an imputer place those cars mid-grid. They are set to "last place + 1" and flagged.

In [4]:
df["pit_lane_start"] = (df["gridPositionText"] == "PL").astype(int)
field = df.groupby("raceId")["gridPositionNumber"].transform("max")
df["grid"] = df["gridPositionNumber"].fillna(field + 1)
print("Pit-lane starts:", int(df["pit_lane_start"].sum()), "| grid still missing:", int(df["grid"].isna().sum()))
df = df.dropna(subset=["grid"])
log("With a usable grid position", df, "pit-lane starts kept as last+1 with a flag")

Pit-lane starts: 128 | grid still missing: 0
With a usable grid position                     5,069 rows   pit-lane starts kept as last+1 with a flag


### 2.3 Form features that only look backwards
How strong are the car and the driver coming into this race? Average points per race over the previous
five races, shifted by one so that today's result is never part of today's feature.

In [5]:
df = df.sort_values(["date", "raceId", "positionDisplayOrder"]).reset_index(drop=True)
team_race = df.groupby(["constructorId", "raceId", "date"], as_index=False)["points"].sum().sort_values("date")
team_race["team_form"] = team_race.groupby("constructorId")["points"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
df = df.merge(team_race[["constructorId", "raceId", "team_form"]], on=["constructorId", "raceId"], how="left", validate="many_to_one")
df["driver_form"] = df.groupby("driverId")["points"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
print("No history yet (first race for a team or driver in the window):", df[["team_form", "driver_form"]].isna().sum().to_dict())

No history yet (first race for a team or driver in the window): {'team_form': 44, 'driver_form': 62}


## 3. Explore

In [6]:
train_mask = df["year"] < TEST_FROM
by_grid = df[df["grid"] <= 20].groupby("grid").agg(points=("points_finish", "mean"), dnf=("dnf", "mean"), n=("grid", "size"))
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar(by_grid.index, by_grid["points"] * 100, color=BLUE, width=0.75)
ax.axvline(10.5, color=GREY, ls="--", lw=1); ax.text(10.7, 96, "points are paid\nto the top 10", fontsize=9, color=GREY, va="top")
ax.set(xlabel="Grid position", ylabel="% of starts that finished in the points", xticks=range(1, 21), ylim=(0, 100),
       title="Every row of the grid costs points probability, most steeply between P7 and P12")
fig.savefig(FIG / "fig1_points_by_grid.png"); plt.close(fig)

In [7]:
cls = df[df["status"] == "classified"]
rows = []
for cid, g in cls.groupby("circuitId"):
    if g["raceId"].nunique() >= 5:
        per_race = g.groupby("raceId").apply(lambda r: spearmanr(r["grid"], r["finish"])[0], include_groups=False)
        rows.append({"circuit": cid, "rho": per_race.mean(), "lo": per_race.quantile(.1), "hi": per_race.quantile(.9), "races": len(per_race), "type": g["circuitType"].iloc[0]})
names = pd.read_csv(DATA / "f1db-circuits.csv").set_index("id")["placeName"]
circ = pd.DataFrame(rows).sort_values("rho").reset_index(drop=True)
circ["circuit"] = circ["circuit"].map(names).fillna(circ["circuit"]).replace({"Sirius": "Sochi"})
fig, ax = plt.subplots(figsize=(7.5, 6.2))
ax.hlines(circ["circuit"], circ["lo"], circ["hi"], color="#c9c7bf", lw=3)
ax.scatter(circ["rho"], circ["circuit"], color=[ORANGE if t == "STREET" else BLUE for t in circ["type"]], zorder=3, s=42)
ax.scatter([], [], color=ORANGE, label="street circuit"); ax.scatter([], [], color=BLUE, label="permanent circuit"); ax.legend(frameon=False, loc="upper left")
ax.set(xlabel="Average rank correlation between grid and finish (1 = the grid is the result)", xlim=(0.2, 1),
       title="Where the grid decides the race (circuits with 5+ races, classified finishers)")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig2_circuit_gridlock.png"); plt.close(fig)
print(circ.round(3).to_string())
print(by_grid.round(3).T.to_string())

        circuit    rho     lo     hi  races    type
0     Zandvoort  0.652  0.345  0.854      5    RACE
1         Sochi  0.669  0.453  0.833      8    ROAD
2     São Paulo  0.684  0.361  0.858     11    RACE
3          Baku  0.698  0.386  0.904      9  STREET
4        Sakhir  0.713  0.636  0.862     13    RACE
5        Austin  0.727  0.582  0.840     11    RACE
6         Imola  0.728  0.628  0.866      5    RACE
7         Monza  0.736  0.564  0.905     12    RACE
8   Silverstone  0.741  0.620  0.927     13    RACE
9           Spa  0.742  0.469  0.957     12    RACE
10     Budapest  0.743  0.584  0.903     12    RACE
11    Abu Dhabi  0.752  0.428  0.945     12    RACE
12     Montreal  0.759  0.614  0.882     10  STREET
13    Spielberg  0.765  0.629  0.881     14    RACE
14    Melbourne  0.766  0.552  0.907     10  STREET
15    Singapore  0.772  0.610  0.898     10  STREET
16     Shanghai  0.796  0.660  0.878      8    RACE
17       Jeddah  0.822  0.678  0.931      5  STREET
18  Mexico C

In [8]:
dnf_year = df.groupby("year")["dnf"].mean() * 100
print(dnf_year.round(1).to_dict())
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.plot(dnf_year.index, dnf_year.values, color=BLUE, marker="o", lw=2)
ax.set(xlabel="Season", ylabel="% of starts not classified", ylim=(0, None), xticks=dnf_year.index, title="Cars became more reliable through the hybrid era")
fig.savefig(FIG / "fig3_dnf_by_year.png"); plt.close(fig)

{2014: 19.1, 2015: 18.1, 2016: 16.2, 2017: 20.7, 2018: 18.1, 2019: 11.4, 2020: 15.1, 2021: 10.8, 2022: 14.2, 2023: 11.0, 2024: 9.5, 2025: 10.9}


## 4. Split by season
Train on 2014 to 2022, test on 2023 to 2025. A random split would let the model see other cars from the
*same race* it is being tested on.

In [9]:
# "gridlock": the circuit's grid-to-finish correlation, measured on TRAINING seasons only (using test seasons would be leakage)
gl = (cls[cls["year"] < TEST_FROM].groupby(["circuitId", "raceId"]).apply(lambda r: spearmanr(r["grid"], r["finish"])[0], include_groups=False)
      .groupby("circuitId").mean().rename("gridlock"))
df = df.merge(gl, on="circuitId", how="left")
cat_cols, num_cols = ["circuitType"], ["grid", "pit_lane_start", "team_form", "driver_form", "gridlock", "round"]
train, test = df[df["year"] < TEST_FROM], df[df["year"] >= TEST_FROM]
y_train, y_test = train["points_finish"], test["points_finish"]
print(f"train {len(train):,} starts ({FIRST}-{TEST_FROM - 1}, {y_train.mean():.1%} points) | test {len(test):,} starts ({TEST_FROM}-{LAST}, {y_test.mean():.1%} points)")
print("Circuits in test with no training history (gridlock unknown):", sorted(test.loc[test["gridlock"].isna(), "circuitId"].unique()))

train 3,681 starts (2014-2022, 49.4% points) | test 1,388 starts (2023-2025, 50.4% points)
Circuits in test with no training history (gridlock unknown): ['las-vegas']


## 5. Models

In [10]:
def logit(num, cats):
    steps = [("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num)]
    if cats:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore"), cats))
    return make_pipeline(ColumnTransformer(steps), LogisticRegression(max_iter=500))


def gbm(num, cats):
    pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cats), ("num", "passthrough", num)])
    return make_pipeline(pre, HistGradientBoostingClassifier(categorical_features=list(range(len(cats))), max_iter=200, learning_rate=0.05, max_depth=3,
                                                             early_stopping=True, validation_fraction=0.2, random_state=42))


def score(name, p, y):
    return {"name": name, "auc": roc_auc_score(y, p), "ap": average_precision_score(y, p), "brier": brier_score_loss(y, p)}


results = [{"name": "Baseline: always the average rate", "auc": 0.5, "ap": float(y_test.mean()), "brier": brier_score_loss(y_test, np.full(len(test), y_train.mean()))}]
rule_p = (test["grid"] <= 10).astype(float) * 0.8 + 0.1                       # the paddock rule of thumb: "starts in the top 10, finishes in the top 10"
results.append(score("Rule of thumb: top-10 start", rule_p, y_test))
fitted = {}
for name, make, num, cats in [("Logistic regression, grid only", logit, ["grid"], []), ("Logistic regression, all features", logit, num_cols, cat_cols),
                              ("Gradient boosting, all features", gbm, num_cols, cat_cols)]:
    m = make(num, cats).fit(train[cats + num], y_train)
    p = m.predict_proba(test[cats + num])[:, 1]
    results.append(score(name, p, y_test)); fitted[name] = (m, cats + num, p)
    print(results[-1])

Xa, Xb, ya, yb = train_test_split(df[cat_cols + num_cols], df["points_finish"], test_size=0.25, random_state=42, stratify=df["points_finish"])
auc_random = roc_auc_score(yb, logit(num_cols, cat_cols).fit(Xa, ya).predict_proba(Xb)[:, 1])
print(f"Random split AUC (logistic, all features) = {auc_random:.3f}")

{'name': 'Logistic regression, grid only', 'auc': 0.8460350913621264, 'ap': 0.8402575193305275, 'brier': 0.15847197331427315}
{'name': 'Logistic regression, all features', 'auc': 0.8716414036544851, 'ap': 0.8631465343028273, 'brier': 0.14583905090108143}
{'name': 'Gradient boosting, all features', 'auc': 0.8679163205980066, 'ap': 0.8639089544987241, 'brier': 0.14801931252206513}


Random split AUC (logistic, all features) = 0.856


In [11]:
r = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(r["name"][::-1], r["auc"][::-1], color=[BLUE, BLUE, BLUE, BLUE, GREY], height=0.65)
for y, v in enumerate(r["auc"][::-1]):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9)
ax.set(xlim=(0.45, 1.0), xlabel=f"ROC AUC on {TEST_FROM}-{LAST} (unseen seasons)", title="Grid position alone gets most of the way; car form adds the rest")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig4_models.png"); plt.close(fig)

In [12]:
best_name = "Logistic regression, all features"
best_m, best_cols, best_p = fitted[best_name]
cal = pd.DataFrame({"p": best_p, "y": y_test.values}); cal["bin"] = pd.qcut(cal["p"], 10, duplicates="drop")
calib = cal.groupby("bin", observed=True).agg(pred=("p", "mean"), actual=("y", "mean"), n=("y", "size"))
print(calib.round(3))
pi = permutation_importance(best_m, test[best_cols], y_test, scoring="roc_auc", n_repeats=20, random_state=1)
nice = {"grid": "Grid position", "team_form": "Team form (last 5 races)", "driver_form": "Driver form (last 5 races)", "gridlock": "Circuit: how much grid decides",
        "circuitType": "Street or permanent", "round": "Round of the season", "pit_lane_start": "Pit-lane start"}
imp = pd.Series(pi.importances_mean, index=best_cols).sort_values(ascending=False)
print(imp.round(4))

                  pred  actual    n
bin                                
(0.0923, 0.154]  0.127   0.065  139
(0.154, 0.207]   0.179   0.115  139
(0.207, 0.276]   0.240   0.180  139
(0.276, 0.361]   0.315   0.254  138
(0.361, 0.471]   0.416   0.417  139
(0.471, 0.587]   0.529   0.561  139
(0.587, 0.74]    0.662   0.739  138
(0.74, 0.853]    0.804   0.878  139
(0.853, 0.913]   0.886   0.899  139
(0.913, 0.981]   0.947   0.935  139


grid              0.2197
team_form         0.0374
driver_form       0.0062
pit_lane_start    0.0010
circuitType       0.0000
round            -0.0000
gridlock         -0.0000
dtype: float64


## 6. The decision: what does a 10-place grid penalty cost, and for whom?
Engine-component penalties can be timed, so a team needs to know the price. The answer depends on the car.
"Front-runner" below means one of the six cars from the three teams with the best form coming into that race.

In [13]:
df["front_runner"] = df["team_form"] >= df.groupby("raceId")["team_form"].transform(lambda s: s.nlargest(6).min())
df["grid_band"] = pd.cut(df["grid"], [0, 5, 10, 15, 30], labels=["P1-5", "P6-10", "P11-15", "P16+"])
band = (df.groupby(["front_runner", "grid_band"], observed=True).agg(p_points=("points_finish", "mean"), mean_points=("points", "mean"), n=("points", "size")).reset_index())
print(band.round(3).to_string())
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for flag, colour, lab in [(True, ORANGE, "front-runners (top 3 teams on form)"), (False, BLUE, "everyone else")]:
    g = band[band["front_runner"] == flag]
    ax.plot(g["grid_band"].astype(str), g["p_points"] * 100, marker="o", lw=2.2, color=colour, label=lab)
    for x, (v, n) in enumerate(zip(g["p_points"], g["n"])):
        ax.annotate(f"{v:.0%}\n(n={n:,})", (x, v * 100), textcoords="offset points", xytext=(0, 9 if flag else -24), ha="center", fontsize=8.5, color="#52514e")
ax.set(xlabel="Starting grid band", ylabel="% of starts finishing in the points", ylim=(-16, 112), yticks=range(0, 101, 20), title="A grid drop costs a midfield car far more than a front-runner")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 0.12))
fig.savefig(FIG / "fig5_penalty_cost.png"); plt.close(fig)

# Does the model agree with what happened to front-runners who started outside the top 10 in the TEST seasons?
chk = test.assign(front_runner=df.loc[test.index, "front_runner"])
chk = chk[chk["front_runner"] & (chk["grid"] >= 11)]
model_check = {"n": int(len(chk)), "actual": float(chk["points_finish"].mean())}
for name, (m, cols, _) in fitted.items():
    model_check[name] = float(m.predict_proba(chk[cols])[:, 1].mean())
print(model_check)

   front_runner grid_band  p_points  mean_points     n
0         False      P1-5     0.835        8.894   254
1         False     P6-10     0.615        3.463   927
2         False    P11-15     0.292        1.230  1143
3         False      P16+     0.110        0.372  1224
4          True      P1-5     0.884       14.579  1001
5          True     P6-10     0.793        8.108   324
6          True    P11-15     0.690        5.330   100
7          True      P16+     0.604        5.542    96
{'n': 70, 'actual': 0.6571428571428571, 'Logistic regression, grid only': 0.27091972939549874, 'Logistic regression, all features': 0.5234211026993429, 'Gradient boosting, all features': 0.6513873054760835}


In [14]:
edges = np.linspace(0, 1, 51)
posh, _ = np.histogram(best_p[y_test.values == 1], bins=edges); negh, _ = np.histogram(best_p[y_test.values == 0], bins=edges)
out = {"n_raw": int(len(res)), "n_model": int(len(df)), "seasons": [FIRST, LAST], "test_from": TEST_FROM, "races": int(df["raceId"].nunique()),
       "coerced_nan": int(pos.isna().sum()), "dns_removed": int(dns.sum()), "pit_lane": int(df["pit_lane_start"].sum()),
       "dnf_rate": float(df["dnf"].mean()), "points_rate": float(df["points_finish"].mean()),
       "train": {"n": int(len(train)), "rate": float(y_train.mean())}, "test": {"n": int(len(test)), "rate": float(y_test.mean())},
       "models": results, "auc_random_split": float(auc_random), "funnel": funnel,
       "by_grid": [{"grid": int(k), "points": float(v.points), "dnf": float(v.dnf), "n": int(v.n)} for k, v in by_grid.iterrows()],
       "circuits": circ.round(4).to_dict("records"), "dnf_year": {int(k): float(v) for k, v in dnf_year.items()},
       "calibration": [{"pred": float(x.pred), "actual": float(x.actual), "n": int(x.n)} for x in calib.itertuples()],
       "importance": [{"name": nice[k], "value": float(v)} for k, v in imp.items()],
       "band": [{"front_runner": bool(v.front_runner), "band": str(v.grid_band), "p_points": float(v.p_points), "mean_points": float(v.mean_points), "n": int(v.n)} for v in band.itertuples()],
       "model_check": model_check,
       "bins": {"pos": posh.tolist(), "neg": negh.tolist()}}
(RES / "metrics.json").write_text(json.dumps(out, indent=1))
print("Saved figures/ and results/metrics.json")

Saved figures/ and results/metrics.json
